In [ ]:
# Dont delete this - this is necessary to run it on colab
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
  from google.colab import drive
  drive.mount('/content/drive')
  %cd /content/drive/MyDrive/Python Code/Paper/scripts/fluxes/wrtds/

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/.shortcut-targets-by-id/1jKcNrw1UEQ_p8vbILfPLdYnIma0uaqCy/Python_Code/Paper/scripts/fluxes/wrtds


This is wherer you can change the paths to input different streamflow data

In [ ]:
import os
refresh_folders = True
flow_path = os.getenv("FLOW_DATA_PATH", "../../../data/streamflow/est_outlet_flow/Downstream_Estimate_All_Rivers_rolling60.csv")
chemistry_path = "../../../data/chemistry/cleaned_data/wq_uncen_all.xlsx"
out_data_path = os.getenv("WRTDS_DATA_ROOT", "../../../data/fluxes/WRTDS/workspace_scaled")
areas_path = "../../../data/geospatial/stats/watershed_area_stats.csv"

In [ ]:
%cat "../../../data/geospatial/stats/watershed_area_stats.csv"

river_name,total_area,excluded_area,pct_excluded
Puelo,9121.633248587017,53.14443206859054,0.5826196978136877
Yelcho,11369.277106927846,2694.2693767649585,23.697807269762215
Palena,13147.563085217787,770.0614652383442,5.857066136492992
Cisnes,5116.698636743262,1276.4596006054553,24.946937297403775
Aysen,12181.393610524234,832.9973753534797,6.838276489430598


In [ ]:
#### Edited from above, seems to work with all functionality as of July 25, 1:15pm
#### 10/17/24 nick edit: commented out the flow_path and brought it up here to show how easy it is to do the scaled version. But i checked the "all rivers Flow.csv"
#### in the flow_scaling/output (downstream) folder, and Puelo seems to be missing. Do you know why?
#### there is an UNCOMMENTED edit above `outliers_removed = []` that deletes the river files automatically, set refresh_folders to False to turn off

#flow_path = "scaled_flow.csv"


import os
from scipy import stats
import numpy as np
import pandas as pd

def mkdir(fpath, cd=False):
    if not os.path.isdir(fpath):
        os.mkdir(fpath)
    if cd:
        os.chdir(fpath)


#pwd = os.getcwd().split("/")


rivers = ["Aysen", "Puelo", "Palena", "Cisnes", "Yelcho"]
# Enter in basin area here
#areas = {"Aysen": 12214.02943, "Puelo": 8926.765273, "Palena": 13142.85061, "Cisnes": 5093.113614, "Yelcho": 11366.91242} # km^2
areas = pd.read_csv(areas_path).set_index("river_name")["total_area"].to_dict()
areas_l = [areas[r] for r in rivers]

# rvrclr = {"Aysen": 'b', "Puelo": 'r', "Palena": 'y', "Cisnes": 'g', "Yelcho": 'm'}
# rvrclr_l = [rvrclr[r] for r in rivers]

# rvrmkr = {"Aysen": 'v', "Puelo": 's', "Palena": '*', "Cisnes": '^', "Yelcho": 'D'}

with pd.ExcelFile(chemistry_path) as DATA:
    Aysen_Chem_raw  = pd.read_excel(DATA, 'Aysen',  skiprows=range(2), comment="#")
    Puelo_Chem_raw  = pd.read_excel(DATA, 'Puelo',  skiprows=range(2), comment="#")
    Cisnes_Chem_raw = pd.read_excel(DATA, 'Cisnes', skiprows=range(2), comment="#")
    Yelcho_Chem_raw = pd.read_excel(DATA, 'Yelcho', skiprows=range(2), comment="#")
    Palena_Chem_raw = pd.read_excel(DATA, 'Palena', skiprows=range(2), comment="#")
    detection_limits = pd.read_excel(DATA, 'detection_limits', index_col="Chem")["Limit"]

raw_chem_dfs = {
    "Aysen": Aysen_Chem_raw,
    "Puelo": Puelo_Chem_raw,
    "Cisnes": Cisnes_Chem_raw,
    "Yelcho": Yelcho_Chem_raw,
    "Palena": Palena_Chem_raw
}

for (r, df) in raw_chem_dfs.items():
    df.name = r

# # average out the replicates
# mean_chem_dfs = dict()
# std_chem_dfs = dict()
# stats_chem_dfs = dict()
# for (r, df) in raw_chem_dfs.items():
#     gb = df.drop(columns=["Team", "Site", "replicate"], errors="ignore").groupby("Date")
#     stats_chem_dfs[r] = gb.agg(["mean", "std"]).swaplevel(0, 1, axis=1)

for df in raw_chem_dfs.values():
    df.set_index("Date", inplace=True)

chem_dfs = dict()
for riv, df in raw_chem_dfs.items():
    newdf = df.drop(columns=["Team", "Site", "replicate"], errors="ignore")
    newdf = newdf.loc[:, newdf.dtypes != "object"].groupby("Date").mean()
    chem_dfs[riv] = newdf

assert not np.any([df.index.duplicated().any() for df in chem_dfs.values()]) # check for duplicate dates

In [ ]:
#flow_path = "franken_flow.csv"
#flow_path = "scaled_flow.csv"
all_flow_df = pd.read_csv(flow_path).set_index("Date")
all_flow_df.index = pd.to_datetime(all_flow_df.index)

daily_flow_dfs = {col: all_flow_df[col].rename("Q").to_frame() for col in all_flow_df}

start_date = pd.to_datetime("2022-02-25")
end_date = pd.to_datetime("2023-12-31")

print(f"Trimming data to the period from {start_date} to {end_date}")

for r, dailydf in daily_flow_dfs.items():
    dailydf = dailydf.sort_values("Date")
    dailydf = dailydf.loc[start_date:end_date]
    print(f"River {r} data range after trimming: {dailydf.index.min()} to {dailydf.index.max()}")
    daily_flow_dfs[r] = dailydf

# for df in chem_dfs.values():
#     df["replicate"] = df.groupby(df.index).cumcount()


#chemicals = ["Li", "Na", "K", "Mg", "Ca", "F", "Cl", "NH4", "NNH4", "SO4", "NOx", "DN", "TN", "DON", "PO4", "DP", "TP", "DOP", "DOC", "dSi", "TSS", "Fe"]

##### This is where you decide which constituents to analyze

chemicals = ["TN", "TP", "Fe", "DOC", "dSi", "TSS"]


######Insert data post prep segment

# identify start and end dates
rng_dfs = dict()
for name, df in chem_dfs.items():

    # print date range
    rng_df = pd.DataFrame(
        data=[
            [df[c].dropna().index[0] for c in chemicals],
            [df[c].dropna().index[-1] for c in chemicals],
        ],
        columns=chemicals,
        index=["start", "end"]
    )

    rng_df["inner"] = [rng_df.loc["start"].max(), rng_df.loc["end"].min()]
    rng_df["outer"] = [rng_df.loc["start"].min(), rng_df.loc["end"].max()]
    assert rng_df.at["start", "inner"] < rng_df.at["end", "inner"] # in case there is no overlap between some of the dates of the chem samples
    assert rng_df.at["start", "outer"] < rng_df.at["end", "outer"] # should be logically unnecessary but whatever
    print(name)
    display(rng_df)

    rng_dfs[name] = rng_df

    # save rng_df -- to be used by a post processing script
    rng_big_df = pd.concat(rng_dfs)
    rng_big_df.index.set_names(["river", "which"], inplace=True)
    rng_big_df.to_csv("chem sample ranges.csv")

#####End data post prep segment

if refresh_folders:
  #### New edit: deleting river folders automatically
  import shutil
  for r in rivers:
    current_folder = out_data_path+r
    if os.path.exists(current_folder):
      print(f"Deleting {r}")
      shutil.rmtree(current_folder)
    else:
      print(f"{r} does not exist")

outliers_removed = []

for r in rivers:
    rp = out_data_path+r+"/"
    mkdir(rp)
    for c in chemicals:
        sp = rp+c+"/"
        mkdir(sp)

        mkdir(sp+"Results")

        with open(rp+r+"_INFO.csv", "w") as FILE:
            FILE.write(
                f"param.units,shortName,paramShortName,constitAbbrev,drainSqKm,station.nm,param.nm,staAbbrev\n\
mg/l,{r},{c},{c},{areas[r]},{r},{c},{r}"
            )

        daily_flow_dfs[r].to_csv(rp+r+"_Q_WRTDS.csv", columns=["Q"])
        wdf = chem_dfs[r][c].to_frame().copy()
        wdf["remarks"] = np.where((wdf[c] == "b.d."), "<", "NA") # for b.d. in remarks
        print("Amt b.d.: ", len(wdf[wdf["remarks"] == "<"]))
        wdf[c] = wdf[c].replace("b.d.", detection_limits[c]) # replce b.d. with the detection limits
        wdf[c] = wdf[c].replace([0, "", None, np.nan], "NA")
        print("Amt NA: ", len(wdf[wdf[c] == "NA"]))
        wdf = wdf[wdf[c] != "NA"]  # Remove rows with "NA" in the chemical concentration column

        mean, std = wdf[c].mean(), wdf[c].std()

 ###### NOTE: THIS IS WHERE OUTLIER THRESHOLDS ARE SET IN TERMS OF STANDARD DEVIATION.
 ###### TO AVOID REMOVING ANY OUTLIERS, SET THE LIMITS TO A HIGH NUMBER TIMES THE STANDARD DEVIATION
 ###### THIS WAS PREVIOUSLY SET TO 3*SD, BUT UPDATED TO 10*SD ON APRIL 10, 2025

        lim1 = mean - 10*std
        lim2 = mean + 10*std

        # Identify and log outliers before removing them
        outliers = wdf[(wdf[c] > lim2) | (wdf[c] < lim1)]
        for index, row in outliers.iterrows():
            outliers_removed.append({
                "River": r,
                "Constituent": c,
                "Date": index,
                "Value": row[c],
                "Standard Deviations from Mean": (row[c] - mean) / std
            })

        # Print the number of outliers found for debugging
        print(f"Outliers found for {r} - {c}: {len(outliers)}")

        # Remove outliers
        wdf = wdf[(wdf[c] > lim1) & (wdf[c] < lim2)]

        # Divide values by 1000 to convert from ug/L to mg/L
        wdf[c] /= 1000

#new TEMPORARY insert 2/5/2025 to trim conc data to available streamflow data date range
        flow_dates = daily_flow_dfs[r].index  # Get available flow dates
        wdf = wdf[wdf.index.isin(flow_dates)]  # Keep only concentration data for dates with flow
#end insert

        wdf.to_csv(sp+r+"_"+c+"_WRTDS.csv", columns=["remarks", c])

# Save outliers to CSV
outliers_df = pd.DataFrame(outliers_removed)
outliers_df.to_csv(out_data_path+"outliers_removed.csv", index=False)

# Verify the length of the trimmed streamflow data
for r, dailydf in daily_flow_dfs.items():
    num_days = (dailydf.index.max() - dailydf.index.min()).days + 1  # Add 1 to include both start and end dates
    print(f"River {r} has {num_days} days of data in the trimmed period.")

# Also verify if the number of leap years in the period
num_leap_years = len([year for year in range(start_date.year, end_date.year + 1) if (year % 4 == 0 and year % 100 != 0) or (year % 400 == 0)])
print(f"Number of leap years in the period: {num_leap_years}")

# Check for any blank entries or NaNs in the flow files
for r, dailydf in daily_flow_dfs.items():
    if dailydf.isnull().values.any():
        print(f"Warning: River {r} has blank entries or NaNs in the flow data.")
    else:
        print(f"River {r} has no blank entries or NaNs in the flow data.")


Trimming data to the period from 2022-02-25 00:00:00 to 2023-12-31 00:00:00
River Aysen data range after trimming: 2022-02-25 00:00:00 to 2023-12-31 00:00:00
River Cisnes data range after trimming: 2022-02-25 00:00:00 to 2023-12-31 00:00:00
River Palena data range after trimming: 2022-02-25 00:00:00 to 2023-12-31 00:00:00
River Puelo data range after trimming: 2022-02-25 00:00:00 to 2023-12-31 00:00:00
River Yelcho data range after trimming: 2022-02-25 00:00:00 to 2023-12-31 00:00:00
Aysen


,TN,TP,Fe,DOC,dSi,TSS,inner,outer
start,2022-03-09,2022-03-09,2022-03-09,2022-03-28,2022-03-09,2022-03-09,2022-03-28,2022-03-09
end,2023-08-18,2023-08-18,2023-03-25,2023-01-26,2023-07-06,2023-08-18,2023-01-26,2023-08-18


Puelo


,TN,TP,Fe,DOC,dSi,TSS,inner,outer
start,2022-03-20,2022-03-20,2022-03-20,2022-03-20,2022-03-20,2022-03-20,2022-03-20,2022-03-20
end,2023-07-07,2023-07-07,2023-04-01,2023-02-04,2023-07-07,2023-12-19,2023-02-04,2023-12-19


Cisnes


...

Yelcho


,TN,TP,Fe,DOC,dSi,TSS,inner,outer
start,2022-03-03,2022-03-03,2022-03-03,2022-03-03,2022-03-03,2022-03-03,2022-03-03,2022-03-03
end,2023-11-30,2023-12-18,2023-05-22,2023-02-05,2023-07-10,2023-12-18,2023-02-05,2023-12-18


Palena


,TN,TP,Fe,DOC,dSi,TSS,inner,outer
start,2022-03-03,2022-03-03,2022-03-03,2022-03-03,2022-03-03,2022-03-03,2022-03-03,2022-03-03
end,2023-06-22,2023-06-22,2023-05-20,2023-06-22,2023-06-22,2023-06-22,2023-05-20,2023-06-22


Aysen does not exist
Puelo does not exist
Palena does not exist
Cisnes does not exist
Yelcho does not exist
Amt b.d.:  0
Amt NA:  0
Outliers found for Aysen - TN: 0
Amt b.d.:  0
Amt NA:  0
Outliers found for Aysen - TP: 0
Amt b.d.:  0
Amt NA:  31
Outliers found for Aysen - Fe: 0
Amt b.d.:  0
Amt NA:  35
Outliers found for Aysen - DOC: 0
Amt b.d.:  0
Amt NA:  1
Outliers found for Aysen - dSi: 0
Amt b.d.:  0
Amt NA:  0
Outliers found for Aysen - TSS: 0
Amt b.d.:  0
Amt NA:  2
Outliers found for Puelo - TN: 0
Amt b.d.:  0
Amt NA:  4
Outliers found for Puelo - TP: 0
Amt b.d.:  0
Amt NA:  36
Outliers found for Puelo - Fe: 0
Amt b.d.:  0
Amt NA:  38
Outliers found for Puelo - DOC: 0
Amt b.d.:  0
Amt NA:  4
Outliers found for Puelo - dSi: 0
Amt b.d.:  0
Amt NA:  3
Outliers found for Puelo - TSS: 0
Amt b.d.:  0
Amt NA:  2
Outliers found for Palena - TN: 0
Amt b.d.:  0
Amt NA:  2
Outliers found for Palena - TP: 0
Amt b.d.:  0
Amt NA:  39
Outliers found for Palena - Fe: 0
Amt b.d.:  0
Amt NA:  4

In [ ]:
detection_limits

,Limit
Chem,
CaCO3,1
pH,1
TSS,2
dSi,3
Li,5
Na,8
NH4,13
K,21
Mg,1
